# Website Scraper

This section creates a **Selenium-based web scraper**.

## Steps

1. Load the website using Selenium

2. Extract the HTML

3. Parse it using BeautifulSoup

4. Remove unnecessary elements

5. Extract clean text

### Architecture

Selenium → BeautifulSoup → Clean Text → LLM

In [ ]:
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from openai import OpenAI
from IPython.display import Markdown, display

# ---------------------------------------------------------
# LLM CLIENTS
# ---------------------------------------------------------

openai_client = OpenAI()

ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)


# ---------------------------------------------------------
# SELENIUM
# ---------------------------------------------------------

def create_driver(headless=True):

    options = Options()

    if headless:
        options.add_argument("--headless=new")

    options.add_argument("--window-size=1920,1080")

    return webdriver.Chrome(options=options)


# ---------------------------------------------------------
# WEBSITE SCRAPER
# ---------------------------------------------------------

def scrape_website(url, max_chars=20_000):

    driver = create_driver()

    try:
        driver.get(url)

        # Wait until page body loads
        WebDriverWait(driver, 10).until(
            lambda d: d.find_element("tag name", "body")
        )

        html = driver.page_source

    finally:
        driver.quit()

    soup = BeautifulSoup(html, "html.parser")

    # Get title
    title = soup.title.get_text(strip=True) if soup.title else ""

    # Remove unnecessary HTML
    unwanted_tags = [
        "script",
        "style",
        "svg",
        "img",
        "input",
        "button",
        "noscript",
        "iframe"
    ]

    for tag in soup(unwanted_tags):
        tag.decompose()

    # Prefer main/article content
    content = (
        soup.find("main")
        or soup.find("article")
        or soup.body
    )

    if content:
        text = content.get_text(
            separator="\n",
            strip=True
        )
    else:
        text = ""

    # Clean blank lines
    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    text = "\n".join(lines)

    return {
        "url": url,
        "title": title,
        "text": text[:max_chars]
    }


# ---------------------------------------------------------
# PROMPTS
# ---------------------------------------------------------

system_prompt = """
You are an assistant that analyzes website content.

Use only the website content provided to you.
Do not invent information.

Provide a concise summary of the important information.
Use clear Markdown.
"""


def messages_for(website):

    user_prompt = f"""
Please summarize this website.

Title: {website["title"]}
URL: {website["url"]}

Website Content:

{website["text"]}
"""

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]


# ---------------------------------------------------------
# SUMMARIZER
# ---------------------------------------------------------

def summarize(website, provider="ollama", model=None):

    if provider == "openai":

        client = openai_client

        if model is None:
            model = "gpt-4.1-mini"

    elif provider == "ollama":

        client = ollama_client

        if model is None:
            model = "gemma3"

    else:
        raise ValueError(
            "provider must be 'openai' or 'ollama'"
        )

    response = client.chat.completions.create(
        model=model,
        messages=messages_for(website)
    )

    return response.choices[0].message.content


# ---------------------------------------------------------
# MAIN FUNCTION YOU CALL
# ---------------------------------------------------------

def scrape(url, provider=None, model=None, max_chars=20_000):

    website = scrape_website(
        url,
        max_chars=max_chars
    )

    # Scrape only
    if provider is None:
        return website

    # Scrape + summarize
    return summarize(
        website,
        provider=provider,
        model=model
    )

In [ ]:
summary = scrape(
    "https://www.anthropic.com/news",
    provider="openai",
    model="gpt-4.1-mini"
)

display(Markdown(summary))

In [ ]:
summary = scrape(

    "https://www.anthropic.com/news",

    provider="ollama"

)
display(Markdown(summary))

In [ ]:
summary = scrape(

    "https://www.anthropic.com/news",

    provider="ollama",

    model="gpt-oss"

)

display(Markdown(summary))